# Notebook 5 — Transfer Learning with ResNet50

Denne notebooken trener en pretrained ResNet50 for multi-label klassifisering av CheXpert-bildene.

Mål:
1. Laste inn de samme preprocessede dataene som i notebook 3 og 4
2. Bygge en ResNet50 med pretrained ImageNet-vekter
3. Trene klassifikasjonshodet med frozen backbone
4. Fine-tune de øverste lagene
5. Evaluere modellen med standardmetrikker
6. Beregne per-label AUROC og PR-AUC
7. Finne bedre decision thresholds per label
8. Sammenligne resultatene med baseline CNN og DenseNet121

Hvorfor denne notebooken er viktig:
- Den gir en tredje modell å sammenligne i rapporten
- ResNet50 er en sterk og mye brukt arkitektur
- Notebooken gir både modellresultater og mer avansert evaluering

In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

from google.colab import drive, userdata

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

mixed_precision.set_global_policy("mixed_float16")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled")
    except RuntimeError as e:
        print(e)

print("TensorFlow version:", tf.__version__)
print("GPU available:", gpus)
print("Mixed precision policy:", mixed_precision.global_policy())

GPU memory growth enabled
TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision policy: <DTypePolicy "mixed_float16">


In [ ]:
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/DAT255_CheXpert_Project")
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DIR exists:", PROCESSED_DIR.exists())

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/DAT255_CheXpert_Project
PROCESSED_DIR exists: True


In [ ]:
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import kagglehub
DATASET_ROOT = Path(kagglehub.dataset_download("ashery/chexpert"))

print("DATASET_ROOT:", DATASET_ROOT)
print("Exists:", DATASET_ROOT.exists())

Using Colab cache for faster access to the 'chexpert' dataset.
DATASET_ROOT: /kaggle/input/chexpert
Exists: True


In [ ]:
LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Pleural Effusion"
]

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
HEAD_EPOCHS = 5
FINETUNE_EPOCHS = 5
AUTOTUNE = tf.data.AUTOTUNE

TRAIN_CSV = PROCESSED_DIR / "train_clean_competition_5_U-Ones.csv"
VALID_CSV = PROCESSED_DIR / "valid_clean_competition_5_U-Ones.csv"
PATH_COL = "Path"
NUM_LABELS = len(LABELS)

print("Labels:", LABELS)
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)

Labels: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']
Image size: (224, 224)
Batch size: 32


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)

train_df.head()

Train shape: (191027, 22)
Valid shape: (202, 22)


,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,...,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,image_path,label_vector,file_exists
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,Frontal,AP,1.0,NaN,0.0,NaN,NaN,...,NaN,0.0,0.0,0.0,NaN,NaN,1.0,/content/dat255_chexpert/data/raw/train/patien...,"[0.0, 0.0, 0.0, 0.0, 0.0]",True
1,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87,Frontal,AP,NaN,NaN,1.0,1.0,NaN,...,NaN,1.0,NaN,1.0,NaN,1.0,NaN,/content/dat255_chexpert/data/raw/train/patien...,"[1.0, 1.0, 1.0, 1.0, 1.0]",True
2,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Frontal,AP,NaN,NaN,0.0,1.0,NaN,...,NaN,0.0,NaN,0.0,NaN,1.0,NaN,/content/dat255_chexpert/data/raw/train/patien...,"[0.0, 0.0, 1.0, 0.0, 0.0]",True
3,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41,Frontal,AP,NaN,NaN,0.0,NaN,NaN,...,NaN,0.0,0.0,0.0,NaN,NaN,NaN,/content/dat255_chexpert/data/raw/train/patien...,"[0.0, 0.0, 0.0, 1.0, 0.0]",True
4,CheXpert-v1.0-small/train/patient00004/study1/...,Female,20,Frontal,PA,1.0,0.0,0.0,NaN,NaN,...,NaN,0.0,NaN,0.0,NaN,NaN,NaN,/content/dat255_chexpert/data/raw/train/patien...,"[0.0, 0.0, 0.0, 0.0, 0.0]",True


In [ ]:
def adjust_path(path_str, dataset_root):
    path_str = str(path_str)
    prefix = "CheXpert-v1.0-small/"
    if path_str.startswith(prefix):
        path_str = path_str[len(prefix):]
    return str(dataset_root / path_str)

train_df["image_path"] = train_df[PATH_COL].apply(lambda p: adjust_path(p, DATASET_ROOT))
valid_df["image_path"] = valid_df[PATH_COL].apply(lambda p: adjust_path(p, DATASET_ROOT))

print("First train image exists:", Path(train_df["image_path"].iloc[0]).exists())
print("First valid image exists:", Path(valid_df["image_path"].iloc[0]).exists())

First train image exists: True
First valid image exists: True


In [ ]:
print("Columns:", train_df.columns.tolist())

print("\nTrain label sums:")
print(train_df[LABELS].sum())

print("\nValid label sums:")
print(valid_df[LABELS].sum())

print("\nMissing labels train:")
print(train_df[LABELS].isna().sum())

print("\nMissing labels valid:")
print(valid_df[LABELS].isna().sum())

Columns: ['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices', 'image_path', 'label_vector', 'file_exists']

Train label sums:
Atelectasis         59583.0
Cardiomegaly        30092.0
Consolidation       37364.0
Edema               61493.0
Pleural Effusion    86477.0
dtype: float64

Valid label sums:
Atelectasis         75.0
Cardiomegaly        66.0
Consolidation       32.0
Edema               42.0
Pleural Effusion    64.0
dtype: float64

Missing labels train:
Atelectasis         0
Cardiomegaly        0
Consolidation       0
Edema               0
Pleural Effusion    0
dtype: int64

Missing labels valid:
Atelectasis         0
Cardiomegaly        0
Consolidation       0
Edema               0
Pleural Effusion    0
dtype: int64


In [ ]:
train_df = train_df[train_df["image_path"].apply(lambda p: Path(p).exists())].copy()
valid_df = valid_df[valid_df["image_path"].apply(lambda p: Path(p).exists())].copy()

print("After path filtering:")
print("Train:", train_df.shape)
print("Valid:", valid_df.shape)

After path filtering:
Train: (191027, 22)
Valid: (202, 22)


## Dataaugmentering

Vi bruker lett dataaugmentering (rotasjon og zoom) for å forbedre generalisering.

Horisontal speiling (horizontal flip) er bevisst ikke brukt, fordi venstre/høyre-orientering i røntgenbilder kan ha klinisk betydning.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.03),
    layers.RandomZoom(0.05),
], name="data_augmentation")
print(train_df[[PATH_COL] + LABELS].head())

                                                Path  Atelectasis  \
0  CheXpert-v1.0-small/train/patient00001/study1/...          0.0   
1  CheXpert-v1.0-small/train/patient00002/study2/...          1.0   
2  CheXpert-v1.0-small/train/patient00002/study1/...          0.0   
3  CheXpert-v1.0-small/train/patient00003/study1/...          0.0   
4  CheXpert-v1.0-small/train/patient00004/study1/...          0.0   

   Cardiomegaly  Consolidation  Edema  Pleural Effusion  
0           0.0            0.0    0.0               0.0  
1           1.0            1.0    1.0               1.0  
2           0.0            1.0    0.0               0.0  
3           0.0            0.0    1.0               0.0  
4           0.0            0.0    0.0               0.0  


In [ ]:
preprocess_input = keras.applications.resnet50.preprocess_input

def load_image(path, labels):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    labels = tf.cast(labels, tf.float32)
    return image, labels

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((
    train_df["image_path"].values,
    train_df[LABELS].values.astype("float32")
))

valid_ds = tf.data.Dataset.from_tensor_slices((
    valid_df["image_path"].values,
    valid_df[LABELS].values.astype("float32")
))

train_ds = (
    train_ds
    .shuffle(buffer_size=len(train_df), seed=SEED)
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

valid_ds = (
    valid_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [ ]:
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Image dtype:", images.dtype)
    print("Label dtype:", labels.dtype)

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32, 5)
Image dtype: <dtype: 'float32'>
Label dtype: <dtype: 'float32'>


## Transfer learning-oppsett

Denne modellen bruker ResNet50 forhåndstrent på ImageNet.

Treningen gjøres i to faser:

1. Vi trener et nytt klassifikasjonshode mens backbone er fryst.
2. Vi låser opp de øverste lagene og finjusterer modellen med lavere læringsrate.ƒ

In [ ]:
def build_resnet50_transfer_model(input_shape=(224, 224, 3), num_labels=5):
    inputs = keras.Input(shape=input_shape)

    x = data_augmentation(inputs)

    base_model = keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )
    base_model.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_labels, activation="sigmoid", dtype="float32")(x)

    model = keras.Model(inputs, outputs, name="resnet50_transfer")
    return model, base_model

model, base_model = build_resnet50_transfer_model(
    input_shape=(224, 224, 3),
    num_labels=NUM_LABELS
)

model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step


Model: "resnet50_transfer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ data_augmentation   │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ data_augmentatio… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c

 Total params: 24,121,733 (92.02 MB)

 Trainable params: 529,925 (2.02 MB)

 Non-trainable params: 23,591,808 (90.00 MB)

## Tapsfunksjon og metrikker

Dette er en multi-label klassifikasjonsoppgave, som betyr at ett røntgenbilde kan ha flere diagnoser samtidig.

Derfor bruker vi:
- sigmoid-aktivering i outputlaget (én sannsynlighet per label)
- Binary Crossentropy som tapsfunksjon

For evaluering bruker vi:
- Binary Accuracy
- ROC-AUC
- PR-AUC
- Precision
- Recall

In [ ]:
metrics = [
    keras.metrics.BinaryAccuracy(name="binary_accuracy"),
    keras.metrics.AUC(name="auc_roc", curve="ROC", multi_label=True),
    keras.metrics.AUC(name="auc_pr", curve="PR", multi_label=True),
    keras.metrics.Precision(name="precision"),
    keras.metrics.Recall(name="recall"),
]

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=metrics
)

## Fase 1: trening av klassifikasjonshode

I første fase holder vi backbone fryst og trener bare det nye klassifikasjonshodet.

Vi bruker callbacks for å:
- lagre beste modell
- stoppe tidlig hvis valideringsytelsen slutter å forbedre seg
- redusere læringsraten ved stagnasjon
- lagre treningslogg

In [ ]:
head_checkpoint_path = MODEL_DIR / "resnet50_head_best.keras"
head_csv_log_path = RESULTS_DIR / "resnet50_head_training_log.csv"

head_callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(head_checkpoint_path),
        monitor="val_auc_roc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_auc_roc",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc_roc",
        mode="max",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.CSVLogger(str(head_csv_log_path))
]

In [ ]:
history_head = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=HEAD_EPOCHS,
    callbacks=head_callbacks,
    verbose=1
)

Epoch 1/5
5970/5970 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - auc_pr: 0.3973 - auc_roc: 0.6503 - binary_accuracy: 0.7294 - loss: 0.5761 - precision: 0.5578 - recall: 0.3245
Epoch 1: val_auc_roc improved from None to 0.79062, saving model to /content/drive/MyDrive/DAT255_CheXpert_Project/models/resnet50_head_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/DAT255_CheXpert_Project/models/resnet50_head_best.keras
5970/5970 ━━━━━━━━━━━━━━━━━━━━ 212s 33ms/step - auc_pr: 0.4135 - auc_roc: 0.6665 - binary_accuracy: 0.7409 - loss: 0.5406 - precision: 0.5971 - recall: 0.3081 - val_auc_pr: 0.5508 - val_auc_roc: 0.7906 - val_binary_accuracy: 0.7446 - val_loss: 0.4938 - val_precision: 0.6479 - val_recall: 0.1649 - learning_rate: 0.0010
Epoch 2/5
5968/5970 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - auc_pr: 0.4331 - auc_roc: 0.6816 - binary_accuracy: 0.7466 - loss: 0.5246 - precision: 0.6270 - recall: 0.2977
Epoch 2: val_auc_roc did not improve from 0.79062

Epoch 2: ReduceLROnPlateau reducin

## Fase 2: finjustering av ResNet50

Etter at klassifikasjonshodet er trent, låser vi opp de øverste lagene i ResNet50-modellen.

Kun de siste lagene finjusteres, mens de tidlige lagene forblir fryst. Dette reduserer beregningskostnad og bevarer nyttige features fra pretraining.

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=metrics
)

print("Number of trainable layers in base model:",
      sum(1 for layer in base_model.layers if layer.trainable))

Number of trainable layers in base model: 50


## Callbacks for finjustering

Under finjusteringen bruker vi egne callbacks for å lagre beste fine-tunede modell og overvåke valideringsytelsen.

In [ ]:
finetune_checkpoint_path = MODEL_DIR / "resnet50_finetuned_best.keras"
finetune_csv_log_path = RESULTS_DIR / "resnet50_finetune_training_log.csv"

finetune_callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(finetune_checkpoint_path),
        monitor="val_auc_roc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_auc_roc",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc_roc",
        mode="max",
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.CSVLogger(str(finetune_csv_log_path))
]

In [ ]:
history_finetune = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=FINETUNE_EPOCHS,
    callbacks=finetune_callbacks,
    verbose=1
)

Epoch 1/5
5970/5970 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - auc_pr: 0.4686 - auc_roc: 0.7111 - binary_accuracy: 0.7564 - loss: 0.5079 - precision: 0.6579 - recall: 0.3175
Epoch 1: val_auc_roc improved from None to 0.84509, saving model to /content/drive/MyDrive/DAT255_CheXpert_Project/models/resnet50_finetuned_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/DAT255_CheXpert_Project/models/resnet50_finetuned_best.keras
5970/5970 ━━━━━━━━━━━━━━━━━━━━ 329s 51ms/step - auc_pr: 0.4870 - auc_roc: 0.7241 - binary_accuracy: 0.7607 - loss: 0.5000 - precision: 0.6653 - recall: 0.3399 - val_auc_pr: 0.6563 - val_auc_roc: 0.8451 - val_binary_accuracy: 0.7762 - val_loss: 0.4644 - val_precision: 0.7677 - val_recall: 0.2724 - learning_rate: 1.0000e-05
Epoch 2/5
5969/5970 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - auc_pr: 0.5207 - auc_roc: 0.7482 - binary_accuracy: 0.7703 - loss: 0.4827 - precision: 0.6844 - recall: 0.3764
Epoch 2: val_auc_roc improved from 0.84509 to 0.84849, saving model to 

## Lagring og evaluering

Etter trening lagrer vi sluttmodellen og evaluerer den på valideringssettet.

Deretter lagrer vi:
- evalueringsmetrikker
- treningshistorikk
- figurer
- prediksjoner
- per-label resultater

In [ ]:
final_model_path = MODEL_DIR / "resnet50_final.keras"
model.save(final_model_path)

print("Best head model saved to:", head_checkpoint_path)
print("Best finetuned model saved to:", finetune_checkpoint_path)
print("Final model saved to:", final_model_path)

In [ ]:
eval_results = model.evaluate(valid_ds, return_dict=True)
eval_results

In [ ]:
metrics_path = RESULTS_DIR / "resnet50_eval_metrics.json"

with open(metrics_path, "w") as f:
    json.dump({k: float(v) for k, v in eval_results.items()}, f, indent=2)

print("Saved metrics to:", metrics_path)

In [ ]:
history_head_df = pd.DataFrame(history_head.history)
history_finetune_df = pd.DataFrame(history_finetune.history)

history_head_df["phase"] = "head_training"
history_finetune_df["phase"] = "fine_tuning"

history_df = pd.concat([history_head_df, history_finetune_df], ignore_index=True)
history_df["epoch_total"] = np.arange(1, len(history_df) + 1)

history_path = RESULTS_DIR / "resnet50_history.csv"
history_df.to_csv(history_path, index=False)

print("Saved history to:", history_path)
history_df.head()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch_total"], history_df["loss"], label="train_loss")
plt.plot(history_df["epoch_total"], history_df["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet50 Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch_total"], history_df["auc_roc"], label="train_auc_roc")
plt.plot(history_df["epoch_total"], history_df["val_auc_roc"], label="val_auc_roc")
plt.xlabel("Epoch")
plt.ylabel("AUC ROC")
plt.title("ResNet50 AUC ROC")
plt.legend()
plt.show()

In [ ]:
y_true = valid_df[LABELS].values.astype("float32")
y_pred = model.predict(valid_ds)

pred_df = valid_df[["image_path"] + LABELS].copy()
for i, label in enumerate(LABELS):
    pred_df[f"{label}_pred"] = y_pred[:, i]

pred_path = RESULTS_DIR / "resnet50_validation_predictions.csv"
pred_df.to_csv(pred_path, index=False)

print("Saved predictions to:", pred_path)
print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)

In [ ]:
per_label_metrics = {}

for i, label in enumerate(LABELS):
    label_true = y_true[:, i]
    label_pred = y_pred[:, i]

    try:
        auc_roc = roc_auc_score(label_true, label_pred)
    except ValueError:
        auc_roc = None

    try:
        auc_pr = average_precision_score(label_true, label_pred)
    except ValueError:
        auc_pr = None

    per_label_metrics[label] = {
        "auc_roc": None if auc_roc is None else float(auc_roc),
        "auc_pr": None if auc_pr is None else float(auc_pr)
    }

per_label_metrics

In [ ]:
per_label_metrics_path = RESULTS_DIR / "resnet50_per_label_metrics.json"

with open(per_label_metrics_path, "w") as f:
    json.dump(per_label_metrics, f, indent=2)

print("Saved per-label metrics to:", per_label_metrics_path)

In [ ]:
def find_best_thresholds(y_true, y_pred, labels, thresholds=np.arange(0.1, 0.91, 0.05)):
    best_rows = []

    for i, label in enumerate(labels):
        label_true = y_true[:, i]
        best_f1 = -1
        best_threshold = 0.5
        best_precision = 0.0
        best_recall = 0.0

        for t in thresholds:
            label_pred_bin = (y_pred[:, i] >= t).astype(int)

            f1 = f1_score(label_true, label_pred_bin, zero_division=0)
            precision = precision_score(label_true, label_pred_bin, zero_division=0)
            recall = recall_score(label_true, label_pred_bin, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_threshold = float(t)
                best_precision = float(precision)
                best_recall = float(recall)

        best_rows.append({
            "label": label,
            "best_threshold": best_threshold,
            "best_f1": float(best_f1),
            "precision_at_best_f1": best_precision,
            "recall_at_best_f1": best_recall
        })

    return pd.DataFrame(best_rows)

threshold_df = find_best_thresholds(y_true, y_pred, LABELS)
threshold_df

In [ ]:
thresholds_path = RESULTS_DIR / "resnet50_best_thresholds.csv"
threshold_df.to_csv(thresholds_path, index=False)

print("Saved thresholds to:", thresholds_path)

In [ ]:
default_rows = []
tuned_rows = []

for i, label in enumerate(LABELS):
    label_true = y_true[:, i]

    pred_default = (y_pred[:, i] >= 0.5).astype(int)
    pred_tuned = (y_pred[:, i] >= threshold_df.loc[threshold_df["label"] == label, "best_threshold"].iloc[0]).astype(int)

    default_rows.append({
        "label": label,
        "f1_default_0.5": f1_score(label_true, pred_default, zero_division=0),
        "precision_default_0.5": precision_score(label_true, pred_default, zero_division=0),
        "recall_default_0.5": recall_score(label_true, pred_default, zero_division=0)
    })

    tuned_rows.append({
        "label": label,
        "f1_tuned": f1_score(label_true, pred_tuned, zero_division=0),
        "precision_tuned": precision_score(label_true, pred_tuned, zero_division=0),
        "recall_tuned": recall_score(label_true, pred_tuned, zero_division=0)
    })

default_df = pd.DataFrame(default_rows)
tuned_df = pd.DataFrame(tuned_rows)

threshold_comparison_df = default_df.merge(tuned_df, on="label")
threshold_comparison_df

In [ ]:
threshold_comparison_path = RESULTS_DIR / "resnet50_threshold_comparison.csv"
threshold_comparison_df.to_csv(threshold_comparison_path, index=False)

print("Saved threshold comparison to:", threshold_comparison_path)

In [ ]:
baseline_metrics_path = RESULTS_DIR / "baseline_cnn_eval_metrics.json"
baseline_per_label_path = RESULTS_DIR / "baseline_cnn_per_label_auc.json"

densenet_metrics_path = RESULTS_DIR / "densenet121_eval_metrics.json"
densenet_per_label_path = RESULTS_DIR / "densenet121_per_label_metrics.json"

with open(baseline_metrics_path, "r") as f:
    baseline_metrics = json.load(f)

with open(baseline_per_label_path, "r") as f:
    baseline_per_label = json.load(f)

with open(densenet_metrics_path, "r") as f:
    densenet_metrics = json.load(f)

with open(densenet_per_label_path, "r") as f:
    densenet_per_label = json.load(f)

In [ ]:
comparison_df = pd.DataFrame([
    {
        "model": "baseline_cnn",
        "auc_roc": baseline_metrics.get("auc_roc"),
        "auc_pr": baseline_metrics.get("auc_pr"),
        "binary_accuracy": baseline_metrics.get("binary_accuracy"),
        "precision": baseline_metrics.get("precision"),
        "recall": baseline_metrics.get("recall"),
    },
    {
        "model": "densenet121_transfer",
        "auc_roc": densenet_metrics.get("auc_roc"),
        "auc_pr": densenet_metrics.get("auc_pr"),
        "binary_accuracy": densenet_metrics.get("binary_accuracy"),
        "precision": densenet_metrics.get("precision"),
        "recall": densenet_metrics.get("recall"),
    },
    {
        "model": "resnet50_transfer",
        "auc_roc": eval_results.get("auc_roc"),
        "auc_pr": eval_results.get("auc_pr"),
        "binary_accuracy": eval_results.get("binary_accuracy"),
        "precision": eval_results.get("precision"),
        "recall": eval_results.get("recall"),
    }
])

comparison_path = RESULTS_DIR / "baseline_vs_densenet_vs_resnet50_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

comparison_df

In [ ]:
plt.figure(figsize=(7, 4))
comparison_df.plot(x="model", y="auc_roc", kind="bar", legend=False)
plt.ylabel("AUC ROC")
plt.title("Model comparison: AUC ROC")
plt.xticks(rotation=0)
plt.show()

plt.figure(figsize=(7, 4))
comparison_df.plot(x="model", y="auc_pr", kind="bar", legend=False)
plt.ylabel("AUC PR")
plt.title("Model comparison: AUC PR")
plt.xticks(rotation=0)
plt.show()

In [ ]:
rows = []

for label in LABELS:
    rows.append({
        "label": label,
        "baseline_auc_roc": baseline_per_label.get(label),
        "densenet_auc_roc": densenet_per_label.get(label, {}).get("auc_roc"),
        "resnet50_auc_roc": per_label_metrics.get(label, {}).get("auc_roc"),
        "resnet50_auc_pr": per_label_metrics.get(label, {}).get("auc_pr"),
    })

per_label_comparison_df = pd.DataFrame(rows)
per_label_comparison_df

In [ ]:
per_label_comparison_path = RESULTS_DIR / "per_label_model_comparison.csv"
per_label_comparison_df.to_csv(per_label_comparison_path, index=False)

print("Saved per-label model comparison to:", per_label_comparison_path)

## Oppsummering

Denne notebooken trener en tredje modell, ResNet50, for multi-label klassifisering av CheXpert-bilder.

Notebooken bidrar med:
- en ny transfer learning-modell
- fair comparison mot baseline CNN og DenseNet121
- standard evalueringsmetrikker
- per-label AUROC og PR-AUC
- threshold tuning for bedre F1 per label

Dette gjør prosjektet sterkere fordi vi kan diskutere:
- forskjellen mellom scratch og transfer learning
- forskjellen mellom DenseNet og ResNet
- betydningen av threshold tuning i et multi-label og ubalansert medisinsk datasett